In [ ]:
# 1. Mount Google Drive to persist checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/vesivuus_checkpoints', exist_ok=True)

# 2. Install required dependencies
!pip install -q monai tifffile imagecodecs kagglehub

In [ ]:
import os

# Configure Kaggle credentials uploaded to Colab
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kagglehub

# Download competition dataset
path = kagglehub.competition_download("vesuvius-challenge-surface-detection")
print("[+] Dataset downloaded:", path)

# Link dataset directory and unpack training source
!rm -rf dataset
!ln -s "{path}" dataset

!unzip -o -q /content/vesivuus_colab.zip -d /content/
print("[+] Code and checkpoints ready!")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Unbuffered stdout (-u) streams live training progress directly in Colab output
!python -u train.py \
  --arch segresnet \
  --patch_size 192 \
  --batch_size 1 \
  --accum_steps 2 \
  --samples_per_volume 1 \
  --all_data \
  --epochs 160 \
  --lr 1e-4 \
  --patience 9999 \
  --save_interval 25 \
  --pretrained checkpoints/segresnet3d_p160_fold2_best.pth \
  --save_dir /content/drive/MyDrive/vesivuus_checkpoints \
  --tag finetune192 \
  --resume